# Hyperspectral Super-Resolution — Training Pipeline

**RRDBNet6x** (modified ESRGAN) for 6× EMIT→fused GT upsampling.

1. Setup & Drive
2. **Configuration** ← edit this cell only
3. Data pipeline (zip-based, pre-filtered)
4. Model & dataset registration
5. Training
6. Post-training evaluation

## 1. Setup

In [ ]:
# ── Install BasicSR + dependencies ──
!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio wandb -q

import sys
sys.path.insert(0, '/content/BasicSR')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, textwrap
os.environ['GH_USER'] = userdata.get('GH_USER')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')

askpass_path = '/tmp/gh_askpass.sh'
with open(askpass_path, 'w') as f:
    f.write(textwrap.dedent('''\
        #!/bin/sh\n\
        case "$1" in\n\
          *Username*) echo "$GH_USER" ;;\n\
          *Password*) echo "$GH_TOKEN" ;;\n\
          *) echo "" ;;\n\
        esac\n\
    '''))
os.chmod(askpass_path, 0o700)
os.environ['GIT_ASKPASS'] = askpass_path
os.environ['GIT_TERMINAL_PROMPT'] = '0'

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git 2>/dev/null || echo 'Repo already cloned'

## 2. Configuration — edit this cell only

In [ ]:
from pathlib import Path
from datetime import datetime


class CFG:
    # ── Data source ──────────────────────────────────────────
    DRIVE_ROOT  = Path('/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches')
    DATA_DATE   = '2026-04-02'

    GT_SOURCE   = 'cnmf'               # 'regression', 'sfim', or 'cnmf'
    MAX_AOIS    = None                  # limit AOI count (None = all)

    # ── Data (zips on Drive, RAM disk for training) ──────────
    RAMDISK       = Path('/content/ramdisk')
    RAMDISK_SIZE  = '140g'
    MIN_ZIPS_TO_START = 10

    # ── Model ────────────────────────────────────────────────
    EXP_NAME    = None                  # None = auto-generate

    NUM_BANDS   = 32
    SCALE       = 6
    GT_SIZE     = 144
    BATCH_SIZE  = 64
    TOTAL_ITER  = 200_000
    LR_RATE     = 2e-4
    NUM_FEAT    = 196
    NUM_BLOCK   = 24
    MILESTONES  = [10_000, 20_000, 45_000, 90_000]

    # ── Resume ───────────────────────────────────────────────
    RESUME_STATE = None
    PRETRAIN_G   = None

    # ── Logging ──────────────────────────────────────────────
    VAL_FREQ        = 1000
    SAVE_CKPT_FREQ  = 5_000
    N_VIS_SAMPLES   = 4
    VIS_BANDS       = (10, 6, 2)
    WANDB_PROJECT   = 'EMIT-S2-SuperResolution'
    WANDB_ENTITY    = None

    # ── Constants (do not edit) ──────────────────────────────
    EMIT_SCALE  = 10_000.0
    EMIT_NODATA = 65535


# ── Derived ───────────────────────────────────────────────────
if CFG.EXP_NAME is None:
    _aoi_tag = f'-{CFG.MAX_AOIS}aoi' if CFG.MAX_AOIS else ''
    CFG.EXP_NAME = f'{CFG.GT_SOURCE}-{CFG.NUM_FEAT}x{CFG.NUM_BLOCK}{_aoi_tag}-{datetime.now():%m%d}'

CFG.DATA_DIR = CFG.DRIVE_ROOT / CFG.DATA_DATE
CFG.EXP_DIR  = CFG.DRIVE_ROOT / CFG.EXP_NAME
CFG.ZIP_DIR  = CFG.DATA_DIR / f'zips_{CFG.GT_SOURCE}'

assert CFG.GT_SIZE % CFG.SCALE == 0, \
    f'GT_SIZE={CFG.GT_SIZE} must be divisible by SCALE={CFG.SCALE}'
assert CFG.ZIP_DIR.exists(), f'Zip dir not found: {CFG.ZIP_DIR}'

print(f'Experiment : {CFG.EXP_NAME}')
print(f'GT source  : {CFG.GT_SOURCE}')
print(f'Max AOIs   : {CFG.MAX_AOIS or "all"}')
print(f'LR crop    : {CFG.GT_SIZE // CFG.SCALE}x{CFG.GT_SIZE // CFG.SCALE}')
print(f'GT crop    : {CFG.GT_SIZE}x{CFG.GT_SIZE}')
print(f'Network    : RRDBNet6x  {CFG.NUM_FEAT}f / {CFG.NUM_BLOCK}b')
print(f'Zip dir    : {CFG.ZIP_DIR}')
print(f'RAM disk   : {CFG.RAMDISK} ({CFG.RAMDISK_SIZE})')
print(f'Output     : {CFG.EXP_DIR}')

In [ ]:
import wandb
wandb.login()
print(f'WandB ready — project: {CFG.WANDB_PROJECT}')

## 3. Data pipeline

In [ ]:
import numpy as np
import zipfile, re, shutil, time, threading
import rasterio
from rasterio.io import MemoryFile
from pathlib import Path


def split_aois(all_aois, seed=42):
    all_aois = sorted(all_aois)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(all_aois))

    if CFG.MAX_AOIS and CFG.MAX_AOIS < len(all_aois):
        perm = perm[:CFG.MAX_AOIS]
        print(f'MAX_AOIS={CFG.MAX_AOIS}: using {len(perm)} of {len(all_aois)} AOIs')

    selected = [all_aois[i] for i in perm]
    n = len(selected)
    n_test = max(1, int(0.15 * n))
    n_val  = max(1, int(0.15 * n))

    test_aois  = set(selected[:n_test])
    val_aois   = set(selected[n_test:n_test + n_val])
    train_aois = set(selected[n_test + n_val:])

    print(f'AOI split: {n} total -> {len(train_aois)} train / {len(val_aois)} val / {len(test_aois)} test')
    return train_aois, val_aois, test_aois


def mount_ramdisk():
    rd = CFG.RAMDISK
    rd.mkdir(parents=True, exist_ok=True)
    import subprocess
    mounts = subprocess.run(['mount'], capture_output=True, text=True).stdout
    if str(rd) in mounts:
        print(f'RAM disk already mounted: {rd}')
    else:
        subprocess.run(
            ['mount', '-t', 'tmpfs', '-o', f'size={CFG.RAMDISK_SIZE}', 'tmpfs', str(rd)],
            check=True
        )
        print(f'RAM disk mounted: {rd} ({CFG.RAMDISK_SIZE})')


class ZipCopier:
    def __init__(self, src_dir, dst_dir):
        self.src_dir = src_dir
        self.dst_dir = dst_dir
        self.dst_dir.mkdir(parents=True, exist_ok=True)
        self.all_zips = sorted(src_dir.glob('*.zip'))
        self.copied = []
        self.lock = threading.Lock()
        self.done = threading.Event()
        self.total = len(self.all_zips)
        self._started = False

    def start(self):
        if self._started:
            return self
        self._started = True
        print(f'Copying {self.total} zips from Drive -> RAM disk (background) ...')
        def _copy():
            for zp in self.all_zips:
                dst = self.dst_dir / zp.name
                if not dst.exists():
                    shutil.copy2(zp, dst)
                with self.lock:
                    self.copied.append(dst)
                    n = len(self.copied)
                    if n % 10 == 0 or n == self.total:
                        print(f'  [{n}/{self.total}] zips copied')
            self.done.set()
        threading.Thread(target=_copy, daemon=True).start()
        return self

    def wait_for(self, min_zips=None):
        if min_zips is None:
            self.done.wait()
            print(f'All {self.total} zips copied.')
        else:
            while True:
                with self.lock:
                    if len(self.copied) >= min(min_zips, self.total):
                        print(f'{len(self.copied)}/{self.total} zips ready (min={min_zips})')
                        return
                time.sleep(0.5)


GT_SUFFIXES = {
    'regression': '_regression_synth.tif',
    'sfim':       '_sfim.tif',
    'cnmf':       '_cnmf.tif',
}

def build_index(zip_dir, gt_source, aoi_filter=None):
    gt_suffix = GT_SUFFIXES[gt_source]
    index = []
    skipped_aoi = 0
    skipped_missing = 0
    skipped_bad = 0

    for zp in sorted(zip_dir.glob('*.zip')):
        scene_key = zp.stem
        aoi = scene_key.split('__')[0]

        if aoi_filter is not None and aoi not in aoi_filter:
            skipped_aoi += 1
            continue

        try:
            zf = zipfile.ZipFile(zp, 'r')
            names = set(zf.namelist())
        except Exception:
            continue

        lr_names = sorted(n for n in names if n.endswith('__emit_b32.tif'))
        for lr_name in lr_names:
            tile_prefix = lr_name.replace('__emit_b32.tif', '')
            tid = int(re.search(r'tile(\d+)', tile_prefix).group(1))
            gt_name = f'{tile_prefix}{gt_suffix}'
            if gt_name not in names:
                skipped_missing += 1
                continue
            try:
                raw = zf.read(gt_name)
                with MemoryFile(raw, filename=gt_name) as memfile:
                    with memfile.open(driver='GTiff') as ds:
                        _ = ds.shape
            except Exception:
                skipped_bad += 1
                continue
            index.append((str(zp), lr_name, gt_name, scene_key, tid))
        zf.close()

    print(f'Index: {len(index)} tiles from {zip_dir}')
    if skipped_aoi:     print(f'  Skipped (AOI filter): {skipped_aoi}')
    if skipped_missing: print(f'  Skipped (no GT): {skipped_missing}')
    if skipped_bad:     print(f'  Skipped (bad file): {skipped_bad}')
    return index


mount_ramdisk()

all_aois = set()
for zp in CFG.ZIP_DIR.glob('*.zip'):
    aoi = zp.stem.split('__')[0]
    all_aois.add(aoi)
train_aois, val_aois, test_aois = split_aois(all_aois)

copier = ZipCopier(CFG.ZIP_DIR, CFG.RAMDISK / 'zips')
copier.start()
copier.wait_for(min_zips=CFG.MIN_ZIPS_TO_START)

## 4. Model & dataset registration

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from basicsr.utils.registry import DATASET_REGISTRY
import numpy as np, random, zipfile
import rasterio
from rasterio.io import MemoryFile
from pathlib import Path


def read_tif_from_zip(zip_path, filename):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        raw = zf.read(filename)
    with MemoryFile(raw) as memfile:
        with memfile.open() as ds:
            return ds.read().astype(np.float32)


if 'PairedZipDataset' in DATASET_REGISTRY._obj_map:
    del DATASET_REGISTRY._obj_map['PairedZipDataset']


@DATASET_REGISTRY.register()
class PairedZipDataset(Dataset):

    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.scale = opt.get('scale', 6)
        self.gt_size = opt.get('gt_size', 144)
        self.emit_scale = opt.get('emit_scale', 10_000.0)
        self.emit_nodata = opt.get('emit_nodata', 65535)
        self.index = opt['_index']
        print(f'[PairedZipDataset] {len(self.index)} pairs, '
              f'scale={self.scale}, gt_size={self.gt_size}')

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        zip_path, lr_name, gt_name, _, _ = self.index[idx]

        try:
            gt = read_tif_from_zip(zip_path, gt_name)
        except rasterio.errors.RasterioIOError:
            return self.__getitem__((idx + 1) % len(self))

        lq = read_tif_from_zip(zip_path, lr_name)

        # Normalize uint16 -> float32 reflectance
        mask = lq == self.emit_nodata
        lq /= self.emit_scale
        lq[mask] = 0.0

        gt /= self.emit_scale
        gt = np.clip(np.nan_to_num(gt, nan=0.0), 0.0, 1.5)

        # Random crop (LR-space first)
        if self.opt.get('phase') == 'train' and self.gt_size:
            _, H_gt, W_gt = gt.shape
            gt_size = self.gt_size
            lq_size = gt_size // self.scale

            top_lq  = random.randint(0, max(0, H_gt // self.scale - lq_size))
            left_lq = random.randint(0, max(0, W_gt // self.scale - lq_size))
            top_gt, left_gt = top_lq * self.scale, left_lq * self.scale

            gt = gt[:, top_gt:top_gt + gt_size, left_gt:left_gt + gt_size]
            lq = lq[:, top_lq:top_lq + lq_size, left_lq:left_lq + lq_size]

            if self.opt.get('use_hflip', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=2).copy()
                lq = np.flip(lq, axis=2).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=1).copy()
                lq = np.flip(lq, axis=1).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.rot90(gt, k=1, axes=(1, 2)).copy()
                lq = np.rot90(lq, k=1, axes=(1, 2)).copy()

        return {
            'lq': torch.from_numpy(lq.copy()).float(),
            'gt': torch.from_numpy(gt.copy()).float(),
            'lq_path': f'{zip_path}::{lr_name}',
            'gt_path': f'{zip_path}::{gt_name}',
        }


print('PairedZipDataset registered.')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB
from basicsr.utils.registry import ARCH_REGISTRY


if 'RRDBNet6x' in ARCH_REGISTRY._obj_map:
    del ARCH_REGISTRY._obj_map['RRDBNet6x']


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    """RRDBNet for 6x upsampling: 2x nearest + 3x nearest = 6x total."""

    def __init__(self, num_in_ch, num_out_ch, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        self.body = nn.ModuleList([RRDB(num_feat, num_grow_ch=num_grow_ch)
                                   for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.conv_up1  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        feat = feat + self.conv_body(body_feat)

        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=3, mode='nearest')))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))

        return out


_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x = torch.randn(1, 32, 20, 20)
_y = _net(_x)
assert _y.shape == (1, 32, 120, 120), f'Expected (1,32,120,120), got {_y.shape}'
del _net, _x, _y
print(f'RRDBNet6x registered.  20x20 -> 120x120 ok')

In [ ]:
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import wandb

from basicsr.models.sr_model import SRModel
from basicsr.utils.registry import MODEL_REGISTRY
import basicsr.utils.img_util as _iu
import basicsr.models.sr_model as _srm


# Patch tensor2img for >4 channel data
importlib.reload(_iu)
_orig_t2i = _iu.tensor2img

def _patched_tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):
    t = tensor[0] if isinstance(tensor, list) else tensor
    if t.ndim >= 3 and t.shape[-3] > 4:
        rgb2bgr = False
    return _orig_t2i(tensor, rgb2bgr=rgb2bgr, out_type=out_type, min_max=min_max)

_iu.tensor2img = _patched_tensor2img
_srm.tensor2img = _patched_tensor2img


def compute_psnr(sr, gt, border=6):
    cb = border
    s = sr[:, cb:-cb, cb:-cb] if cb > 0 else sr
    g = gt[:, cb:-cb, cb:-cb] if cb > 0 else gt
    mse = np.mean((s - g) ** 2)
    if mse < 1e-10:
        return 100.0
    return float(10 * np.log10(1.0 / mse))


def compute_sam(sr, gt, border=6):
    cb = border
    s = sr[:, cb:-cb, cb:-cb] if cb > 0 else sr
    g = gt[:, cb:-cb, cb:-cb] if cb > 0 else gt
    dot = np.sum(s * g, axis=0)
    norm_s = np.linalg.norm(s, axis=0).clip(1e-8)
    norm_g = np.linalg.norm(g, axis=0).clip(1e-8)
    cos_sim = np.clip(dot / (norm_s * norm_g), -1 + 1e-7, 1 - 1e-7)
    sam_map = np.degrees(np.arccos(cos_sim))
    return float(np.mean(sam_map))


def compute_ergas(sr, gt, scale=6, border=6):
    cb = border
    s = sr[:, cb:-cb, cb:-cb] if cb > 0 else sr
    g = gt[:, cb:-cb, cb:-cb] if cb > 0 else gt
    band_rmse = np.sqrt(np.mean((s - g) ** 2, axis=(1, 2)))
    band_mean = np.mean(g, axis=(1, 2)).clip(1e-8)
    ergas = 100.0 / scale * np.sqrt(np.mean((band_rmse / band_mean) ** 2))
    return float(ergas)


def compute_per_band_correlation(sr, gt, border=6):
    cb = border
    s = sr[:, cb:-cb, cb:-cb] if cb > 0 else sr
    g = gt[:, cb:-cb, cb:-cb] if cb > 0 else gt
    C = s.shape[0]
    corrs = np.zeros(C)
    for c in range(C):
        sf = s[c].flatten()
        gf = g[c].flatten()
        if np.std(sf) < 1e-8 or np.std(gf) < 1e-8:
            corrs[c] = 0.0
        else:
            corrs[c] = np.corrcoef(sf, gf)[0, 1]
    return corrs


def compute_all_metrics(sr, gt, bic, scale=6, border=6):
    return {
        'sr_psnr':  compute_psnr(sr, gt, border),
        'sr_sam':   compute_sam(sr, gt, border),
        'sr_ergas': compute_ergas(sr, gt, scale, border),
        'bic_psnr':  compute_psnr(bic, gt, border),
        'bic_sam':   compute_sam(bic, gt, border),
        'bic_ergas': compute_ergas(bic, gt, scale, border),
    }


def to_rgb(cube, bands=CFG.VIS_BANDS):
    rgb = np.stack([cube[b] for b in bands], axis=-1)
    pos = rgb[rgb > 0]
    if len(pos) > 0:
        lo, hi = np.percentile(pos, [2, 98])
    else:
        lo, hi = 0.0, 1.0
    return np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)


def make_main_figure(lq_np, sr_np, gt_np, bic_np, metrics, bands, aoi_label=''):
    sr_rgb  = to_rgb(sr_np, bands)
    gt_rgb  = to_rgb(gt_np, bands)
    bic_rgb = to_rgb(bic_np, bands)
    rmse_map_sr  = np.sqrt(np.mean((sr_np  - gt_np) ** 2, axis=0))
    rmse_map_bic = np.sqrt(np.mean((bic_np - gt_np) ** 2, axis=0))
    dot   = np.sum(sr_np * gt_np, axis=0)
    n_sr  = np.linalg.norm(sr_np, axis=0).clip(1e-8)
    n_gt  = np.linalg.norm(gt_np, axis=0).clip(1e-8)
    sam_map = np.degrees(np.arccos(np.clip(dot / (n_sr * n_gt), -1 + 1e-7, 1 - 1e-7)))
    sr_bic_diff = np.sqrt(np.mean((sr_np - bic_np) ** 2, axis=0))
    rmse_vmax = max(rmse_map_sr.max(), rmse_map_bic.max()) * 0.9

    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    pfx = f'[{aoi_label}] ' if aoi_label else ''
    axes[0, 0].imshow(bic_rgb)
    axes[0, 0].set_title(f'{pfx}Bicubic\nPSNR={metrics["bic_psnr"]:.2f}  SAM={metrics["bic_sam"]:.2f}  ERGAS={metrics["bic_ergas"]:.1f}', fontsize=9)
    axes[0, 1].imshow(sr_rgb)
    axes[0, 1].set_title(f'{pfx}SR\nPSNR={metrics["sr_psnr"]:.2f}  SAM={metrics["sr_sam"]:.2f}  ERGAS={metrics["sr_ergas"]:.1f}', fontsize=9)
    axes[0, 2].imshow(gt_rgb)
    axes[0, 2].set_title(f'{pfx}GT ({CFG.GT_SOURCE})', fontsize=9)
    im1 = axes[1, 0].imshow(rmse_map_sr, cmap='hot', vmin=0, vmax=rmse_vmax)
    axes[1, 0].set_title('SR RMSE (per pixel)', fontsize=9)
    plt.colorbar(im1, ax=axes[1, 0], fraction=0.046)
    im2 = axes[1, 1].imshow(sam_map, cmap='hot', vmin=0)
    axes[1, 1].set_title('SR SAM (degrees)', fontsize=9)
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)
    im3 = axes[1, 2].imshow(sr_bic_diff, cmap='coolwarm', vmin=0)
    axes[1, 2].set_title('|SR - Bicubic| RMSE', fontsize=9)
    plt.colorbar(im3, ax=axes[1, 2], fraction=0.046)
    for ax in axes[0]: ax.axis('off')
    for ax in axes[1]: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    return fig


def make_perband_figure(sr_np, gt_np, bic_np, border=6, aoi_label=''):
    cb = border
    sr_c  = sr_np[:, cb:-cb, cb:-cb]  if cb > 0 else sr_np
    gt_c  = gt_np[:, cb:-cb, cb:-cb]  if cb > 0 else gt_np
    bic_c = bic_np[:, cb:-cb, cb:-cb] if cb > 0 else bic_np
    C = sr_c.shape[0]
    bands = np.arange(C)
    rmse_sr  = np.sqrt(np.mean((sr_c  - gt_c) ** 2, axis=(1, 2)))
    rmse_bic = np.sqrt(np.mean((bic_c - gt_c) ** 2, axis=(1, 2)))
    corr_sr  = compute_per_band_correlation(sr_np, gt_np, border)
    corr_bic = compute_per_band_correlation(bic_np, gt_np, border)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    pfx = f'[{aoi_label}] ' if aoi_label else ''
    ax1.plot(bands, rmse_bic, 'o-', ms=4, color='#FF9800', label='Bicubic', alpha=0.8, linewidth=1.5)
    ax1.plot(bands, rmse_sr,  's-', ms=4, color='#2196F3', label='SR',      alpha=0.8, linewidth=1.5)
    ax1.fill_between(bands, rmse_sr, rmse_bic, where=(rmse_sr < rmse_bic), color='#4CAF50', alpha=0.15, label='SR wins')
    ax1.fill_between(bands, rmse_sr, rmse_bic, where=(rmse_sr >= rmse_bic), color='#F44336', alpha=0.15, label='Bicubic wins')
    ax1.set_ylabel('RMSE'); ax1.set_title(f'{pfx}Per-band RMSE: SR vs Bicubic'); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.set_ylim(bottom=0)
    ax2.plot(bands, corr_bic, 'o-', ms=4, color='#FF9800', label='Bicubic', alpha=0.8, linewidth=1.5)
    ax2.plot(bands, corr_sr,  's-', ms=4, color='#2196F3', label='SR',      alpha=0.8, linewidth=1.5)
    ax2.set_ylabel('Pearson r'); ax2.set_xlabel('Band index'); ax2.set_title(f'{pfx}Per-band Correlation with GT'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.05); ax2.set_xticks(bands)
    plt.tight_layout()
    return fig


def make_zoom_figure(sr_np, gt_np, bic_np, bands, crop_frac=0.35, aoi_label=''):
    _, H, W = gt_np.shape
    ch, cw = int(H * crop_frac), int(W * crop_frac)
    y0 = (H - ch) // 2; x0 = (W - cw) // 2
    sr_rgb  = to_rgb(sr_np[:,  y0:y0+ch, x0:x0+cw], bands)
    gt_rgb  = to_rgb(gt_np[:,  y0:y0+ch, x0:x0+cw], bands)
    bic_rgb = to_rgb(bic_np[:, y0:y0+ch, x0:x0+cw], bands)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    pfx = f'[{aoi_label}] ' if aoi_label else ''
    axes[0].imshow(bic_rgb, interpolation='nearest'); axes[0].set_title(f'{pfx}Bicubic (zoom)')
    axes[1].imshow(sr_rgb, interpolation='nearest');  axes[1].set_title(f'{pfx}SR (zoom)')
    axes[2].imshow(gt_rgb, interpolation='nearest');  axes[2].set_title(f'{pfx}GT (zoom)')
    for ax in axes: ax.axis('off')
    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.01, wspace=0.05)
    return fig


def select_vis_indices(dataset, n_vis):
    n = len(dataset)
    n_vis = min(n_vis, n)
    aoi_groups = {}
    for idx in range(n):
        aoi = dataset.index[idx][3].split('__')[0]
        aoi_groups.setdefault(aoi, []).append(idx)
    if len(aoi_groups) >= n_vis:
        aoi_list = sorted(aoi_groups.keys())
        step = max(1, len(aoi_list) // n_vis)
        return [aoi_groups[a][len(aoi_groups[a]) // 2] for a in aoi_list[::step][:n_vis]]
    indices = []
    for aoi in sorted(aoi_groups.keys()):
        tiles = aoi_groups[aoi]
        per_aoi = max(1, n_vis // len(aoi_groups))
        step = max(1, len(tiles) // per_aoi)
        indices.extend(tiles[::step][:per_aoi])
    return sorted(set(indices))[:n_vis]


print('Metrics & visualization helpers loaded.')

In [ ]:
from collections import OrderedDict
from basicsr.losses import build_loss

if 'WandBSRModel' in MODEL_REGISTRY._obj_map:
    del MODEL_REGISTRY._obj_map['WandBSRModel']

@MODEL_REGISTRY.register()
class WandBSRModel(SRModel):

    def optimize_parameters(self, current_iter):
        self.optimizer_g.zero_grad()
        self.output = self.net_g(self.lq)

        l_total = torch.tensor(0.0, device=self.device)
        loss_dict = OrderedDict()

        if self.cri_pix:
            l_pix = self.cri_pix(self.output, self.gt)
            l_total = l_total + l_pix
            loss_dict['l_pix'] = l_pix

        l_total.backward()

        if self.opt['train'].get('grad_clip'):
            torch.nn.utils.clip_grad_norm_(
                self.net_g.parameters(),
                self.opt['train']['grad_clip']['max_norm']
            )

        self.optimizer_g.step()
        self.log_dict = self.reduce_loss_dict(loss_dict)

        if self.ema_decay > 0:
            self.model_ema(decay=self.ema_decay)

    def nondist_validation(self, dataloader, current_iter, tb_logger, save_img):
        super().nondist_validation(dataloader, current_iter, tb_logger, save_img)

        if wandb.run is None:
            return

        dataset = dataloader.dataset
        scale = self.opt.get('scale', CFG.SCALE)
        border = scale

        all_sr_psnr, all_sr_sam, all_sr_ergas = [], [], []
        all_bic_psnr, all_bic_sam, all_bic_ergas = [], [], []
        all_sr_corr, all_bic_corr = [], []
        cached = {}

        for idx in range(len(dataset)):
            data = dataset[idx]
            lq = data['lq'].unsqueeze(0).to(self.device)
            with torch.no_grad():
                sr = self.net_g(lq)
            lq_np = data['lq'].numpy()
            sr_np = sr.squeeze(0).cpu().numpy()
            gt_np = data['gt'].numpy()
            bic_np = F.interpolate(
                torch.from_numpy(lq_np[None]), scale_factor=scale,
                mode='bicubic', align_corners=False
            ).squeeze(0).numpy()

            m = compute_all_metrics(sr_np, gt_np, bic_np, scale, border)
            all_sr_psnr.append(m['sr_psnr'])
            all_sr_sam.append(m['sr_sam'])
            all_sr_ergas.append(m['sr_ergas'])
            all_bic_psnr.append(m['bic_psnr'])
            all_bic_sam.append(m['bic_sam'])
            all_bic_ergas.append(m['bic_ergas'])
            all_sr_corr.append(compute_per_band_correlation(sr_np, gt_np, border))
            all_bic_corr.append(compute_per_band_correlation(bic_np, gt_np, border))
            cached[idx] = (lq_np, sr_np, gt_np, bic_np, m)

        n_val = len(dataset)
        wandb.log({
            'val/sr_psnr_mean':   np.mean(all_sr_psnr),
            'val/sr_sam_mean':    np.mean(all_sr_sam),
            'val/sr_ergas_mean':  np.mean(all_sr_ergas),
            'val/bic_psnr_mean':  np.mean(all_bic_psnr),
            'val/bic_sam_mean':   np.mean(all_bic_sam),
            'val/bic_ergas_mean': np.mean(all_bic_ergas),
            'val/delta_psnr':  np.mean(all_sr_psnr)  - np.mean(all_bic_psnr),
            'val/delta_sam':   np.mean(all_bic_sam)   - np.mean(all_sr_sam),
            'val/delta_ergas': np.mean(all_bic_ergas) - np.mean(all_sr_ergas),
            'val/sr_corr_mean':  np.mean(all_sr_corr),
            'val/bic_corr_mean': np.mean(all_bic_corr),
            'val/sr_wins_psnr':  sum(s > b for s, b in zip(all_sr_psnr, all_bic_psnr)) / n_val,
            'val/sr_wins_sam':   sum(s < b for s, b in zip(all_sr_sam, all_bic_sam)) / n_val,
            'val/sr_wins_ergas': sum(s < b for s, b in zip(all_sr_ergas, all_bic_ergas)) / n_val,
            'global_step': current_iter,
        })

        n_vis = min(CFG.N_VIS_SAMPLES, len(dataset))
        vis_indices = select_vis_indices(dataset, n_vis)

        images = {}
        for i, idx in enumerate(vis_indices):
            lq_np, sr_np, gt_np, bic_np, metrics = cached[idx]
            aoi_label = dataset.index[idx][3].split('__')[0].replace('aoi_', '')

            fig_main = make_main_figure(lq_np, sr_np, gt_np, bic_np, metrics, CFG.VIS_BANDS, aoi_label=aoi_label)
            images[f'val_vis/tile_{i}_main'] = wandb.Image(fig_main)
            plt.close(fig_main)

            fig_bands = make_perband_figure(sr_np, gt_np, bic_np, border, aoi_label)
            images[f'val_vis/tile_{i}_perband'] = wandb.Image(fig_bands)
            plt.close(fig_bands)

            fig_zoom = make_zoom_figure(sr_np, gt_np, bic_np, CFG.VIS_BANDS, crop_frac=0.35, aoi_label=aoi_label)
            images[f'val_vis/tile_{i}_zoom'] = wandb.Image(fig_zoom)
            plt.close(fig_zoom)

        wandb.log({**images, 'global_step': current_iter})

print('WandBSRModel registered.')

## 5. Training

In [ ]:
print('Waiting for zip copy to finish ...')
copier = ZipCopier(CFG.ZIP_DIR, CFG.RAMDISK / 'zips')
copier.start()
copier.wait_for(min_zips=None)

zip_local = CFG.RAMDISK / 'zips'

train_index = build_index(zip_local, CFG.GT_SOURCE, aoi_filter=train_aois)
val_index   = build_index(zip_local, CFG.GT_SOURCE, aoi_filter=val_aois)
test_index  = build_index(zip_local, CFG.GT_SOURCE, aoi_filter=test_aois)

In [ ]:
import yaml
from pathlib import Path

config = {
    'name': CFG.EXP_NAME,
    'model_type': 'WandBSRModel',
    'scale': CFG.SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedZipDataset',
            '_index': train_index,
            'gt_size': CFG.GT_SIZE,
            'scale': CFG.SCALE,
            'emit_scale': CFG.EMIT_SCALE,
            'emit_nodata': CFG.EMIT_NODATA,
            'use_hflip': True,
            'use_rot': True,
            'phase': 'train',
            'num_worker_per_gpu': 10,
            'batch_size_per_gpu': CFG.BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedZipDataset',
            '_index': val_index,
            'scale': CFG.SCALE,
            'emit_scale': CFG.EMIT_SCALE,
            'emit_nodata': CFG.EMIT_NODATA,
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': CFG.NUM_BANDS,
        'num_out_ch': CFG.NUM_BANDS,
        'num_feat': CFG.NUM_FEAT,
        'num_block': CFG.NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': str(CFG.PRETRAIN_G) if CFG.PRETRAIN_G else None,
        'strict_load_g': True,
        'resume_state': str(CFG.RESUME_STATE) if CFG.RESUME_STATE else None,
        'experiments_root': str(CFG.EXP_DIR),
        'models': str(CFG.EXP_DIR / 'models'),
        'training_states': str(CFG.EXP_DIR / 'training_states'),
        'log': str(CFG.EXP_DIR),
        'visualization': str(CFG.EXP_DIR / 'visualization'),
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': CFG.LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': CFG.MILESTONES,
            'gamma': 0.5,
        },
        'total_iter': CFG.TOTAL_ITER,
        'warmup_iter': -1,
        'grad_clip': {'type': 'norm', 'max_norm': 5.0},
        'pixel_opt': {'type': 'L1Loss', 'loss_weight': 1.0, 'reduction': 'mean'},
    },

    'val': {
        'val_freq': CFG.VAL_FREQ,
        'save_img': False,
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': CFG.SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': CFG.SAVE_CKPT_FREQ,
        'use_tb_logger': True,
        'wandb': {
            'project': CFG.WANDB_PROJECT,
            'resume_id': None,
        },
    },

    'dist_params': {'backend': 'nccl', 'port': 29500},
}

config_path = Path('/content/BasicSR/options/train') / f'{CFG.EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'Config: {config_path}')
print(f'  GT crop:   {CFG.GT_SIZE}x{CFG.GT_SIZE}  ->  LR crop: {CFG.GT_SIZE // CFG.SCALE}x{CFG.GT_SIZE // CFG.SCALE}')
print(f'  Network:   RRDBNet6x  {CFG.NUM_FEAT}f / {CFG.NUM_BLOCK}b')
print(f'  Iters:     {CFG.TOTAL_ITER:,}   batch: {CFG.BATCH_SIZE}')
print(f'  LR:        {CFG.LR_RATE}  milestones: {CFG.MILESTONES}')
print(f'  Val freq:  {CFG.VAL_FREQ}   Ckpt freq: {CFG.SAVE_CKPT_FREQ}')

In [ ]:
# Bicubic baseline (sanity check)
import torch, torch.nn.functional as F, numpy as np

psnrs = []
for zip_path, lr_name, gt_name, _, _ in val_index:
    lq = read_tif_from_zip(zip_path, lr_name).astype(np.float32)
    gt = read_tif_from_zip(zip_path, gt_name).astype(np.float32)

    mask = lq == CFG.EMIT_NODATA
    lq /= CFG.EMIT_SCALE; lq[mask] = 0.0
    gt /= CFG.EMIT_SCALE; gt = np.clip(np.nan_to_num(gt, nan=0.0), 0.0, 1.5)

    bic = F.interpolate(torch.from_numpy(lq[None]).float(),
                        scale_factor=6, mode='bicubic',
                        align_corners=False).squeeze(0).numpy()
    cb = CFG.SCALE
    mse = np.mean((gt[:, cb:-cb, cb:-cb] - bic[:, cb:-cb, cb:-cb]) ** 2)
    if mse > 0:
        psnrs.append(10 * np.log10(1.0 / mse))

print(f'Bicubic baseline (val, crop_border={CFG.SCALE}): {np.mean(psnrs):.2f} +/- {np.std(psnrs):.2f} dB')

In [ ]:
import sys
sys.path.insert(0, '/content/BasicSR')
sys.argv = ['train.py', '-opt', str(config_path)]

from basicsr.train import train_pipeline
train_pipeline('/content/BasicSR')

## 6. Post-training evaluation

In [ ]:
import re, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm


def find_checkpoint(exp_dir, exp_name):
    search = [Path(exp_dir) / 'models',
              Path(exp_dir) / exp_name / 'models',
              Path('/content/BasicSR/experiments') / exp_name / 'models']
    ckpts = []
    for d in search:
        if d.exists():
            ckpts.extend(d.glob('net_g_*.pth'))
    numbered = [(p, int(m.group(1))) for p in ckpts
                if (m := re.search(r'(\d+)', p.stem))]
    if not numbered:
        print(f'No checkpoints found in: {[str(d) for d in search]}')
        return None
    numbered.sort(key=lambda x: x[1])
    print(f'Checkpoint: {numbered[-1][0]}')
    return numbered[-1][0]


def load_model(ckpt_path, device='cuda'):
    net = RRDBNet6x(num_in_ch=CFG.NUM_BANDS, num_out_ch=CFG.NUM_BANDS,
                    num_feat=CFG.NUM_FEAT, num_block=CFG.NUM_BLOCK)
    state = torch.load(ckpt_path, map_location='cpu')
    key = 'params_ema' if 'params_ema' in state else 'params' if 'params' in state else None
    net.load_state_dict(state[key] if key else state)
    print(f'Loaded {"EMA" if key == "params_ema" else "regular"} weights')
    return net.to(device).eval()


def evaluate_on_split(index, ckpt_path=None, split_name='test'):
    ckpt = Path(ckpt_path) if ckpt_path else find_checkpoint(CFG.EXP_DIR, CFG.EXP_NAME)
    if ckpt is None:
        return
    net = load_model(ckpt)
    scale = CFG.SCALE
    border = scale

    sr_psnrs, bic_psnrs, sr_sams, bic_sams = [], [], [], []

    for zip_path, lr_name, gt_name, scene_key, tid in tqdm(index, desc=f'Eval {split_name}'):
        lq = read_tif_from_zip(zip_path, lr_name).astype(np.float32)
        gt = read_tif_from_zip(zip_path, gt_name).astype(np.float32)

        mask = lq == CFG.EMIT_NODATA
        lq /= CFG.EMIT_SCALE; lq[mask] = 0.0
        gt /= CFG.EMIT_SCALE; gt = np.clip(np.nan_to_num(gt, nan=0.0), 0.0, 1.5)

        with torch.no_grad():
            sr = net(torch.from_numpy(lq[None]).float().cuda()).squeeze(0).cpu().numpy()
        bic = F.interpolate(torch.from_numpy(lq[None]).float(),
                            scale_factor=scale, mode='bicubic',
                            align_corners=False).squeeze(0).numpy()

        sr_psnrs.append(compute_psnr(sr, gt, border))
        bic_psnrs.append(compute_psnr(bic, gt, border))
        sr_sams.append(compute_sam(sr, gt, border))
        bic_sams.append(compute_sam(bic, gt, border))

    print(f'\n{split_name} results ({len(index)} tiles):')
    print(f'  SR:      PSNR={np.mean(sr_psnrs):.2f} dB   SAM={np.mean(sr_sams):.2f} deg')
    print(f'  Bicubic: PSNR={np.mean(bic_psnrs):.2f} dB   SAM={np.mean(bic_sams):.2f} deg')
    print(f'  Delta:   PSNR=+{np.mean(sr_psnrs) - np.mean(bic_psnrs):.2f} dB   SAM=-{np.mean(bic_sams) - np.mean(sr_sams):.2f} deg')


evaluate_on_split(test_index)